# Adversarial TTS — Harvard Sentences Experiment

Runs the adversarial optimization for every sentence in a configurable slice of the Harvard sentence list.

Results are stored under `outputs/results/sentence_XXX/run_Y/` and can be aggregated afterwards with `Scripts/aggregate_results.py`.

## 1. Setup

In [ ]:
%%bash
git clone https://github.com/Vorgesetzter/StyleTTS2
cd StyleTTS2
pip install -r requirements.txt
sudo apt-get install espeak-ng
git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LJSpeech
mkdir -p Audio
mv StyleTTS2-LJSpeech/Models/* Audio/
rm -rf StyleTTS2-LJSpeech

In [ ]:
%cd StyleTTS2

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # Must be set before cuBLAS initializes

import signal
import argparse
import numpy as np
import torch
import matplotlib.pyplot as plt
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

from Datastructures.dataclass import ModelData
from Objectives.FitnessObjective import FitnessObjective
from Trainer.EnvironmentLoader import EnvironmentLoader
from Trainer.AdversarialTrainer import AdversarialTrainer
from Trainer.RunLogger import RunLogger
from Trainer.VectorManipulator import VectorManipulator
from Optimizer.pymoo_optimizer import PymooOptimizer
from pymoo.algorithms.moo.nsga2 import NSGA2

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Available GPUs: {torch.cuda.device_count()}")

In [ ]:
%%bash
git pull

## 2. Configuration

Edit `harvard_sentences_start`, `harvard_sentences_end`, and the optimization parameters below.

In [ ]:
# =============================================================================
# Harvard Sentence Lists (IEEE, Rothauser et al. 1969)
# Source: https://www.cs.columbia.edu/~hgs/audio/harvard.html
# =============================================================================
HARVARD_SENTENCES = [
    # List 1
    "The birch canoe slid on the smooth planks.",
    "Glue the sheet to the dark blue background.",
    "It's easy to tell the depth of a well.",
    "These days a chicken leg is a rare dish.",
    "Rice is often served in round bowls.",
    "The juice of lemons makes fine punch.",
    "The box was thrown beside the parked truck.",
    "The hogs were fed chopped corn and garbage.",
    "Four hours of steady work faced us.",
    "A large size in stockings is hard to sell.",

    # List 2
    "The boy was there when the sun rose.",
    "A rod is used to catch pink salmon.",
    "The source of the huge river is the clear spring.",
    "Kick the ball straight and follow through.",
    "Help the woman get back to her feet.",
    "A pot of tea helps to pass the evening.",
    "Smoky fires lack flame and heat.",
    "The soft cushion broke the man's fall.",
    "The salt breeze came across from the sea.",
    "The girl at the booth sold fifty bonds.",

    # List 3
    "The small pup gnawed a hole in the sock.",
    "The fish twisted and turned on the bent hook.",
    "Press the pants and sew a button on the vest.",
    "The swan dive was far short of perfect.",
    "The beauty of the view stunned the young boy.",
    "Two blue fish swam in the tank.",
    "Her purse was full of useless trash.",
    "The colt reared and threw the tall rider.",
    "It snowed, rained, and hailed the same morning.",
    "Read verse out loud for pleasure.",

    # List 4
    "Hoist the load to your left shoulder.",
    "Take the winding path to reach the lake.",
    "Note closely the size of the gas tank.",
    "Wipe the grease off his dirty face.",
    "Mend the coat before you go out.",
    "The wrist was badly strained and hung limp.",
    "The stray cat gave birth to kittens.",
    "The young girl gave no clear response.",
    "The meal was cooked before the bell rang.",
    "What joy there is in living.",

    # List 5
    "A king ruled the state in the early days.",
    "Ship maps are different from those for planes.",
    "Dimes showered down from all sides.",
    "The two met while playing on the sand.",
    "The ink stain dried on the finished page.",
    "The walled town was seized without a fight.",
    "The lease ran out in sixteen weeks.",
    "A tame squirrel makes a nice pet.",
    "The horn of the car woke the sleeping cop.",
    "The heart beat strongly and with firm strokes.",

    # List 6
    "The pearl was worn in a thin silver ring.",
    "The fruit peel was cut in thick slices.",
    "The Navy attacked the big task force.",
    "See the cat glaring at the scared mouse.",
    "There are more than two factors here.",
    "The hat brim was wide and too droopy.",
    "The lawyer tried to lose his case.",
    "The grass curled around the fence post.",
    "Cut the pie into large parts.",
    "Men strive but seldom get rich.",

    # List 7
    "Always close the barn door tight.",
    "He ran half way to the hardware store.",
    "The clock struck to mark the third period.",
    "A small creek cut across the field.",
    "Cars and busses stalled in snow drifts.",
    "The set of china hit the floor with a crash.",
    "This is a grand season for hikes on the road.",
    "The dune rose from the edge of the water.",
    "Those words were the cue for the actor to leave.",
    "A yacht slid around the point into the bay.",

    # List 8
    "The two boys' voices echoed in the hall.",
    "The gold vase is both rare and costly.",
    "The knife was hung inside its bright sheath.",
    "The rarest spice comes from the Far East.",
    "The roof should be tilted at a sharp slant.",
    "A siege will take a lot out of the troops.",
    "The green moss grew over the stone well.",
    "The shelves were bare of both jam or crackers.",
    "Every word and phrase he speaks is true.",
    "The bombs left most of the town in ruins.",

    # List 9
    "Stop whistling and watch the boys march.",
    "Jerk the rope and the bell rings weakly.",
    "A clean neck means a neat collar.",
    "A new wax protects the deep scratch.",
    "The cup cracked and spilled its contents.",
    "Guess the results from the first scores.",
    "A salt pickle tastes fine with ham.",
    "The just claim got the right verdict.",
    "The purple tie was ten years old.",
    "The tree top waved in a graceful way.",

    # List 10
    "The spot on the blotter was made by green ink.",
    "Mud was spattered on the front of his white shirt.",
    "The cigar burned a hole in the desk top.",
    "The empty flask stood on the tin tray.",
    "A speedy man can beat this track mark.",
    "He broke a new shoelace that day.",
    "The coffee in the mug was too hot to drink.",
    "The birch trees were bare and lonely.",
    "The petals fall with every light breeze.",
    "Bring your problems to the wise chief.",
]

# Use in your experiment
import random
random.seed(42)
test_sentences = random.sample(HARVARD_SENTENCES, 100)  # All 100

print(f"Harvard sentences loaded: {len(HARVARD_SENTENCES)} entries")

In [ ]:
# =============================================================================
# Experiment Configuration
# =============================================================================

# --- Sentence range (1-indexed, inclusive) ---
harvard_sentences_start = 1
harvard_sentences_end   = 10

# --- Runs per sentence ---
loop_count = 2

# --- Optimization parameters ---
num_generations    = 100
pop_size           = 100
batch_size         = 100   # -1 for full batch
iv_scalar          = 0.5
size_per_phoneme   = 1
subspace_optimization = False

# --- Attack mode and objectives ---
mode       = "NOISE_UNTARGETED"  # TARGETED | UNTARGETED | NOISE_UNTARGETED
target_text = ""
objectives = "PESQ=0.2, SET_OVERLAP=0.5"

# --- Optional output flags ---
save_spectrograms = True  # set True to save mel-spectrogram PNGs per run
save_graphs       = True  # set True to save hypervolume / Pareto graphs per run

# Validate range
assert 1 <= harvard_sentences_start <= harvard_sentences_end <= len(HARVARD_SENTENCES), (
    f"Sentence range [{harvard_sentences_start}, {harvard_sentences_end}] is out of bounds "
    f"for HARVARD_SENTENCES (len={len(HARVARD_SENTENCES)})."
)

print(f"Sentences : {harvard_sentences_start} – {harvard_sentences_end} "
      f"({harvard_sentences_end - harvard_sentences_start + 1} sentences)")
print(f"Runs/sentence: {loop_count}")
print(f"Total runs: {(harvard_sentences_end - harvard_sentences_start + 1) * loop_count}")

## 3. Load Models (once)

Models are loaded a single time and reused across all sentences and runs.

In [ ]:
loader = EnvironmentLoader(device)
tts_model, asr_model = loader.load_required_models()
print("Models loaded.")

## 4. Run Experiment

Outer loop: sentences. Inner loop: independent runs, each with a fresh random noise direction.

In [ ]:
import signal

def _handle_sigterm(signum, frame):
    raise KeyboardInterrupt

signal.signal(signal.SIGTERM, _handle_sigterm)

try:
    for sentence_id in range(harvard_sentences_start, harvard_sentences_end + 1):
        sentence_text = HARVARD_SENTENCES[sentence_id - 1]
        print(f"\n{'='*60}")
        print(f"[Sentence {sentence_id}] {sentence_text}")
        print(f"{'='*60}")

        for run_id in range(loop_count):
            print(f"\n  [Run {run_id + 1}/{loop_count}]")

            # --- Per-run setup: fresh noise direction for every run ---
            args = argparse.Namespace(
                ground_truth_text=sentence_text,
                target_text=target_text,
                loop_count=1,
                num_generations=num_generations,
                pop_size=pop_size,
                iv_scalar=iv_scalar,
                size_per_phoneme=size_per_phoneme,
                batch_size=batch_size,
                notify=False,
                subspace_optimization=subspace_optimization,
                mode=mode,
                objectives=objectives,
            )
            config_data = loader.load_configuration(args)

            audio_gt, audio_target, audio_embedding_gt, audio_embedding_target = loader.generate_audio_data(
                config_data.mode, config_data.text_gt, config_data.text_target, tts_model
            )

            objectives_dict = loader.initialize_objectives(
                active_objectives=config_data.active_objectives,
                model_data=ModelData(tts_model=tts_model, asr_model=asr_model),
                text_gt=config_data.text_gt,
                text_target=config_data.text_target,
                mode=config_data.mode,
                audio_gt=audio_gt,
            )

            vector_manipulator = VectorManipulator(audio_embedding_gt, audio_embedding_target.h_text, config_data)

            trainer = AdversarialTrainer(
                tts_model, asr_model, config_data.thresholds, objectives_dict, vector_manipulator, device
            )
            logger = RunLogger(
                config_data.active_objectives, tts_model, asr_model, vector_manipulator, device
            )

            optimizer = PymooOptimizer(
                bounds=(0, 1),
                algorithm=NSGA2,
                algo_params={"pop_size": pop_size},
                num_objectives=len(config_data.active_objectives),
                solution_shape=(
                    int(audio_embedding_gt.input_length.detach().cpu().item()), size_per_phoneme,
                ),
            )

            fitness_data = None
            try:
                fitness_data, archive_data, generation_count, elapsed_time_total = trainer.run_full_iteration(
                    optimizer, num_generations, pop_size, batch_size
                )
            except KeyboardInterrupt:
                # Save whatever was collected before the interrupt, then stop.
                print("  [!] Saving partial results for this run...")
                if fitness_data:
                    logger.save_results_run(
                        optimizer=optimizer,
                        fitness_data=fitness_data,
                        archive_data=archive_data,
                        generation_count=generation_count,
                        elapsed_time_total=elapsed_time_total,
                        audio_gt=audio_gt,
                        audio_target=audio_target,
                        config_data=config_data,
                        sentence_id=sentence_id,
                        run_id=run_id,
                        num_generations=num_generations,
                        save_spectrograms=save_spectrograms,
                        save_graphs=save_graphs,
                    )
                raise  # propagate to outer try/except to stop the experiment

            logger.save_results_run(
                optimizer=optimizer,
                fitness_data=fitness_data,
                archive_data=archive_data,
                generation_count=generation_count,
                elapsed_time_total=elapsed_time_total,
                audio_gt=audio_gt,
                audio_target=audio_target,
                config_data=config_data,
                sentence_id=sentence_id,
                run_id=run_id,
                num_generations=num_generations,
                save_spectrograms=save_spectrograms,
                save_graphs=save_graphs,
            )

            torch.cuda.empty_cache()

except KeyboardInterrupt:
    print("\n[!] Experiment stopped early. All completed runs have been saved.")

finally:
    # Always generate the zip, even if the experiment was cancelled
    import shutil, os
    on_kaggle = os.path.exists('/kaggle/working')
    if on_kaggle:
        shutil.make_archive('/kaggle/working/outputs', 'zip', '/kaggle/working/StyleTTS2/outputs')
        print("Created /kaggle/working/outputs.zip")
    else:
        shutil.make_archive('/content/outputs', 'zip', '/content/StyleTTS2/outputs')
        print("Created /content/outputs.zip")
print("\n[Done]")


## 5. Aggregate Results

Collects all `run_summary.json` files into `outputs/all_results.csv` and `outputs/all_results.json`.

In [ ]:
import subprocess
result = subprocess.run(
    ["python", "Scripts/aggregate_results.py",
     "--results_dir", "outputs/results",
     "--output_dir", "outputs"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 6. Download Results

Creates a zip of the outputs folder.

**On Colab:** uncomment `files.download`.  
**On Kaggle:** zip appears in the Output tab after the session ends.

In [ ]:
import shutil
import os

on_kaggle = os.path.exists('/kaggle/working')

if on_kaggle:
    shutil.make_archive('/kaggle/working/outputs', 'zip', '/kaggle/working/StyleTTS2/outputs')
    print("Created /kaggle/working/outputs.zip — available in the Output tab.")
else:
    shutil.make_archive('/content/outputs', 'zip', '/content/StyleTTS2/outputs')
    print("Created /content/outputs.zip")
    # from google.colab import files
    # files.download('/content/outputs.zip')